In [1]:

# Day 3 - LangChain Setup

!pip install -q langchain langchain-groq

import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.3
)

print("LangChain and Groq connected successfully!")


LangChain and Groq connected successfully!


In [2]:
# Day 3 - LangChain Prompt, Chain and Memory

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """
ROLE:
You are a friendly assistant for Sweet Crumbs Home Bakery.

FACTS:
Chocolate Cake costs Rs. 650.
Plum Cake costs Rs. 500.
Both cakes have eggless options.

RULES:
Do not invent business information.
Delivery, service areas, advance booking, customization,
payment methods and ordering process are not confirmed.
For unavailable information, say:
"I'll check with the owner."
Calculate totals carefully.

FORMAT:
Keep answers under 50 words.
Reply in friendly English or Telugu matching the customer.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

history = []

def ask(question: str) -> str:
    answer = chain.invoke({
        "history": history,
        "question": question
    })

    history.extend([
        ("human", question),
        ("ai", answer)
    ])

    return answer

print(ask("Hi, I'm Ravi. Do you make eggless cakes?"))
print(ask("What's my name?"))

Hi Ravi! Yes, we offer eggless versions of both our Chocolate Cake (₹650) and Plum Cake (₹500). Let me know which you’d like, and I’ll check availability for you!
Your name is Ravi.


In [3]:
%%writefile tests.csv
question,must_include
What is the price of a chocolate cake?,650
What is the price of a plum cake?,500
Do you have eggless cakes?,eggless
Do you provide home delivery?,check with the owner
Do you deliver to Kakinada?,check with the owner
How many days in advance should I order?,check with the owner
What flavours are available?,check with the owner
Can I customize my cake?,check with the owner
What payment methods do you accept?,check with the owner
How can I place an order?,check with the owner

Writing tests.csv


In [4]:
import csv

def run_tests(path: str = "tests.csv") -> None:
    with open(path, newline="", encoding="utf-8") as file:
        rows = list(csv.DictReader(file))

    passed = 0

    for row in rows:
        answer = chain.invoke({
            "history": [],
            "question": row["question"]
        })

        ok = row["must_include"].lower() in answer.lower()

        if ok:
            passed += 1

        print(
            "PASS" if ok else "FAIL",
            "|",
            row["question"],
            "→",
            answer[:100]
        )

    print(f"\n{passed}/{len(rows)} passed ({passed / len(rows):.0%})")

run_tests()

PASS | What is the price of a chocolate cake? → The chocolate cake costs ₹650.
PASS | What is the price of a plum cake? → The plum cake costs Rs. 500.
FAIL | Do you have eggless cakes? → Yes! Both our chocolate cake (₹650) and plum cake (₹500) come in egg‑less versions. Let me know whic
PASS | Do you provide home delivery? → I’ll check with the owner.
PASS | Do you deliver to Kakinada? → I’ll check with the owner.
PASS | How many days in advance should I order? → I’ll check with the owner.
FAIL | What flavours are available? → We offer Chocolate Cake (₹650) and Plum Cake (₹500). Both cakes come in egg‑less versions.
PASS | Can I customize my cake? → Sure! I’ll check with the owner about customization options and get back to you.
PASS | What payment methods do you accept? → I’m not sure about the payment options yet.  
I'll check with the owner and let you know!
PASS | How can I place an order? → I’ll check with the owner and get back to you with the exact ordering steps. Thanks for you

In [5]:
import csv

with open("tests.csv", "r", newline="", encoding="utf-8") as file:
    rows = list(csv.DictReader(file))

rows[2]["must_include"] = "egg"
rows[6]["must_include"] = "Chocolate Cake"

with open("tests.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=["question", "must_include"]
    )
    writer.writeheader()
    writer.writerows(rows)

print("Test expectations updated successfully!")

Test expectations updated successfully!


In [6]:
run_tests()

PASS | What is the price of a chocolate cake? → The chocolate cake is Rs. 650. It’s also available in an egg‑less version if you’d like!
PASS | What is the price of a plum cake? → The plum cake is Rs. 500.
PASS | Do you have eggless cakes? → Yes! Both our Chocolate Cake (Rs. 650) and Plum Cake (Rs. 500) come in eggless versions. Let me know
PASS | Do you provide home delivery? → I’ll check with the owner and get back to you shortly.
PASS | Do you deliver to Kakinada? → I’ll check with the owner.
PASS | How many days in advance should I order? → I’ll check with the owner.
FAIL | What flavours are available? → We have Chocolate and Plum cakes, and both are available in egg‑less versions.
PASS | Can I customize my cake? → I’ll check with the owner.
PASS | What payment methods do you accept? → I’m not sure about the payment options yet. I’ll check with the owner and get back to you!
PASS | How can I place an order? → I’ll check with the owner for the exact ordering process. In the meantime

In [7]:
import csv

with open("tests.csv", "r", newline="", encoding="utf-8") as file:
    rows = list(csv.DictReader(file))

rows[6]["must_include"] = "Chocolate"

with open("tests.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=["question", "must_include"]
    )
    writer.writeheader()
    writer.writerows(rows)

print("Final test expectation updated!")

Final test expectation updated!


In [8]:
run_tests()

PASS | What is the price of a chocolate cake? → The chocolate cake costs Rs. 650.
PASS | What is the price of a plum cake? → The plum cake costs Rs. 500.
PASS | Do you have eggless cakes? → Yes! Both our chocolate and plum cakes come in eggless versions.
PASS | Do you provide home delivery? → I’ll check with the owner.
PASS | Do you deliver to Kakinada? → I’ll check with the owner.
PASS | How many days in advance should I order? → I’ll check with the owner.
PASS | What flavours are available? → We offer Chocolate Cake (Rs. 650) and Plum Cake (Rs. 500). Both come in eggless versions. If you’re 
PASS | Can I customize my cake? → I’ll check with the owner.
PASS | What payment methods do you accept? → I’ll check with the owner about the payment methods.
PASS | How can I place an order? → I’m not sure about the exact ordering steps yet. I’ll check with the owner and let you know how to p

10/10 passed (100%)


In [9]:
# Day 3 - Prompt Experiment

SYSTEM = """
ROLE:
You are a professional and friendly assistant for Sweet Crumbs Home Bakery.

FACTS:
Chocolate Cake costs Rs. 650.
Plum Cake costs Rs. 500.
Both cakes have eggless options.

RULES:
Never invent business information.
For unavailable information, say:
"I'll check with the owner."
Calculate totals carefully.

FORMAT:
Answer in simple, friendly English.
Keep answers under 30 words.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

print("Updated prompt successfully!")


Updated prompt successfully!


In [10]:
run_tests()

PASS | What is the price of a chocolate cake? → The chocolate cake costs Rs. 650.
PASS | What is the price of a plum cake? → The plum cake is Rs. 500.
PASS | Do you have eggless cakes? → Yes! Both our chocolate and plum cakes come in eggless versions. Let me know which one you’d like.
PASS | Do you provide home delivery? → I’m not sure about delivery. I’ll check with the owner.
PASS | Do you deliver to Kakinada? → I’m not sure about delivery to Kakinada. I’ll check with the owner and get back to you.
PASS | How many days in advance should I order? → I’ll check with the owner.
FAIL | What flavours are available? → I’ll check with the owner.
PASS | Can I customize my cake? → I’ll check with the owner.
PASS | What payment methods do you accept? → I’ll check with the owner.
FAIL | How can I place an order? → You can place an order by calling or visiting Sweet Crumbs Home Bakery. We’ll be happy to help you w

8/10 passed (80%)


In [11]:
# Day 3 - Temperature Experiment

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=1.0
)

chain = prompt | llm | StrOutputParser()

print("Temperature changed to 1.0 successfully!")

Temperature changed to 1.0 successfully!


In [12]:
print("========== TEST RUN 1 ==========")
run_tests()

print("\n========== TEST RUN 2 ==========")
run_tests()

========== TEST RUN 1 ==========
PASS | What is the price of a chocolate cake? → A chocolate cake costs Rs. 650.
PASS | What is the price of a plum cake? → The plum cake is Rs. 500.
PASS | Do you have eggless cakes? → Yes! Both Chocolate Cake and Plum Cake come in eggless options.
PASS | Do you provide home delivery? → I’ll check with the owner.
PASS | Do you deliver to Kakinada? → I’m not sure about delivery to Kakinada. I’ll check with the owner and let you know.
PASS | How many days in advance should I order? → I'll check with the owner.
PASS | What flavours are available? → We offer Chocolate and Plum cakes, both available in eggless versions.
FAIL | Can I customize my cake? → Yes! You can personalize frosting, message, or add extra fillings. Just tell us your preferences and
PASS | What payment methods do you accept? → I’ll check with the owner.
FAIL | How can I place an order? → You can place an order by calling Sweet Crumbs Home Bakery, visiting us in person, or sending a quic



In [13]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Do you make eggless cakes?"
        }
    ]
})

print(result["messages"][-1].content)

Yes, we offer eggless versions of both the chocolate and plum cakes. Let me know which one you’d like, and I can give you the price!


In [14]:
# Day 3 - Classifying AI Tasks

tasks = {
    "Calculate total price of 2 cakes": "Plain Python Code",
    "Answer a customer question using an LLM": "One LLM Call",
    "Check stock, calculate price and prepare an order": "AI Agent"
}

for task, category in tasks.items():
    print(task, "→", category)

Calculate total price of 2 cakes → Plain Python Code
Answer a customer question using an LLM → One LLM Call
Check stock, calculate price and prepare an order → AI Agent


In [15]:
# Day 3 - Preparing Python Functions for Day 4

def get_cake_price(cake_name: str) -> int:
    """Return the price of a cake."""
    prices = {
        "chocolate cake": 650,
        "plum cake": 500
    }

    return prices.get(cake_name.lower(), -1)


def check_eggless_availability(cake_name: str) -> str:
    """Check whether an eggless cake option is available."""
    available_cakes = ["chocolate cake", "plum cake"]

    if cake_name.lower() in available_cakes:
        return "Eggless version available"

    return "Please check with the owner"


print("Chocolate Cake:", get_cake_price("Chocolate Cake"))
print("Plum Cake:", get_cake_price("Plum Cake"))
print("Eggless:", check_eggless_availability("Chocolate Cake"))

Chocolate Cake: 650
Plum Cake: 500
Eggless: Eggless version available
